In [ ]:
from math_verify import parse, verify
from math_verify.parser import ExprExtractionConfig, LatexExtractionConfig


# GSM8K answer column typically looks like this:
gsm8k_answer = """Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May.
So she sold 48/2 = 24 clips in May.
In total she sold 48 + 24 = 72 clips.
#### 72"""

# Step 1: Extract the gold answer from the GSM8K "#### <number>" format
def extract_gsm8k_gold(answer_text: str) -> str:
    """Pull the numeric answer after ####"""
    parts = answer_text.split("####")
    if len(parts) > 1:
        return parts[-1].strip().replace(",", "")  # handle "1,000" -> "1000"
    return answer_text.strip()

gold_str = extract_gsm8k_gold(gsm8k_answer)
print(f"Extracted gold: {gold_str}")  # "72"

# Step 2: Parse both gold and model prediction with math-verify
gold = parse(gold_str, extraction_config=[ExprExtractionConfig()])

# Suppose your model produced this output:
model_output = "Let me work through this step by step.\n48 / 2 = 24\n48 + 24 = 72\nThe answer is #### 72"

# For model output, use both extraction configs for robustness
# answer = parse(model_output, extraction_config=[ExprExtractionConfig()])
answer = parse(model_output, extraction_config=[LatexExtractionConfig(), ExprExtractionConfig()])

# Step 3: Verify
result = verify(gold, answer)
print(answer)
print(f"Match: {result}")  # True

Extracted gold: 72
[]
Match: False


In [ ]:
help

In [2]:
# example of sentence transformers from HuggingFace page
from sentence_transformers import SentenceTransformer

# 1. Load a pretrained Sentence Transformer model
model = SentenceTransformer("all-MiniLM-L6-v2")

# The sentences to encode
sentences = [
    "The weather is lovely today.",
    "It's so sunny outside!",
    "He drove to the stadium.",
]

# 2. Calculate embeddings by calling model.encode()
embeddings = model.encode(sentences)
print(embeddings.shape)
# [3, 384]

# 3. Calculate the embedding similarities
similarities = model.similarity(embeddings, embeddings)
print(similarities)
# tensor([[1.0000, 0.6660, 0.1046],
#         [0.6660, 1.0000, 0.1411],
#         [0.1046, 0.1411, 1.0000]])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


(3, 384)
tensor([[1.0000, 0.6660, 0.1046],
        [0.6660, 1.0000, 0.1411],
        [0.1046, 0.1411, 1.0000]])


In [ ]:
# Cosine similarity
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Load HuggingFace embedding model
model = SentenceTransformer("Qwen/Qwen2.5-Coder-0.5B-Instruct")

# -----------------------------
# Example dataset
# -----------------------------
query = "The cat is sleeping on the sofa"

candidates = [
    "A cat is lying on the couch",
    "The dog is running in the park",
    "A feline is resting on a sofa",
]

# -----------------------------
# Embedding function
# -----------------------------
def get_similarity(a, b):
    emb = model.encode([a, b])
    return cosine_similarity([emb[0]], [emb[1]])[0][0]

# -----------------------------
# ZERO-SHOT (no examples)
# -----------------------------
print("\n=== 0-SHOT SIMILARITY ===")
for c in candidates:
    score = get_similarity(query, c)
    print(f"{query} <-> {c}")
    print("Similarity:", round(score, 4))
    print()

# -----------------------------
# FEW-SHOT EXAMPLES
# -----------------------------
examples_3shot = [
    ("The cat is sleeping", "A cat is resting"),
    ("Dogs are barking", "A dog is making noise"),
    ("Bird is flying", "A bird is in the air"),
]

examples_5shot = examples_3shot + [
    ("Man is eating food", "A person is having a meal"),
    ("Car is moving fast", "Vehicle is driving quickly"),
]

def build_prompt(query, examples):
    prompt = ""
    for a, b in examples:
        prompt += f"Example:\n{a}\n{b}\n\n"
    prompt += f"Target:\n{query}"
    return prompt

# -----------------------------
# 3-SHOT
# -----------------------------
print("\n=== 3-SHOT SIMILARITY ===")
for c in candidates:
    q = build_prompt(query, examples_3shot)
    score = get_similarity(q, c)
    print(f"{c}")
    print("Similarity:", round(score, 4))
    print()

# -----------------------------
# 5-SHOT
# -----------------------------
print("\n=== 5-SHOT SIMILARITY ===")
for c in candidates:
    q = build_prompt(query, examples_5shot)
    score = get_similarity(q, c)
    print(f"{c}")
    print("Similarity:", round(score, 4))
    print()

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


=== 0-SHOT SIMILARITY ===
The cat is sleeping on the sofa <-> A cat is lying on the couch
Similarity: 0.999

The cat is sleeping on the sofa <-> The dog is running in the park
Similarity: 0.9986

The cat is sleeping on the sofa <-> A feline is resting on a sofa
Similarity: 0.9984


=== 3-SHOT SIMILARITY ===
A cat is lying on the couch
Similarity: 0.9907

The dog is running in the park
Similarity: 0.9897

A feline is resting on a sofa
Similarity: 0.9887


=== 5-SHOT SIMILARITY ===
A cat is lying on the couch
Similarity: 0.9912

The dog is running in the park
Similarity: 0.9908

A feline is resting on a sofa
Similarity: 0.9894



In [ ]:
# Temprature
"""
Temperature is a parameter that controls the randomness of the output generated by a language model. 
It is used during the sampling process to determine how likely the model is to choose less probable tokens.
A higher temperature will result in lower probability, i.e more creative outputs. A lower temperature will result in higher probability, i.e more predictable outputs. 

"""
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# ----------------------------
# Load model (change if needed)
# ----------------------------
model_name = "Qwen/Qwen2.5-0.5B-Instruct" 

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

# ----------------------------
# Prompt
# ----------------------------
prompt = "Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?\n"

inputs = tokenizer(prompt, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}
# ----------------------------
# Function to generate with temperature
# ----------------------------
def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,          # IMPORTANT for temperature
            temperature=temp,
            top_p=0.95,
            repetition_penalty=1.1
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

# ----------------------------
# Run experiments
# ----------------------------
temperatures = [0.01] * 100
for t in temperatures:
    print("\n" + "="*50)
    print(f"TEMPERATURE = {t}")
    print("="*50)
    print(generate_with_temperature(t))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


TEMPERATURE = 0.01
Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?
To find the total number of pieces of fruit, we need to calculate the number of oranges and the number of nectarines separately, then add them together.

First, let's calculate the number of oranges:
There are 12 crates, and each crate contains 150 oranges.
So, the total number of oranges is \(12 \times 150 = 18

TEMPERATURE = 0.01
Solve: There are 12 crates that each contain 150 oranges. There are 16 boxes that each hold 30 nectarines. How many pieces of fruit are in the crates and the boxes in total?
To find the total number of pieces of fruit, we need to calculate the number of oranges and the number of nectarines separately, then add them together.

First, let's calculate the number of oranges:
- There are 12 crates.
- Each crate contains 150 oranges.
So, the total number of oranges is \(12 

In [6]:
# model is too small
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

prompt = """Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=100,
            do_sample=True,
            temperature=temp,
            top_p=0.95
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

for t in [0.3, 0.7, 1.0, 1.3]:
    print("\n" + "="*50)
    print("TEMPERATURE:", t)
    print("="*50)
    print(generate_with_temperature(t))

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


TEMPERATURE: 0.3
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?
assistant
To determine the total number of pieces of fruit, we need to calculate the number of oranges and nectarines separately and then sum them up.

First, let's calculate the number of oranges:
- There are 12 crates.
- Each crate contains 150 oranges.
- Therefore, the total number of oranges is:
  \[
  12 \times 150 = 1800
  \]

Next, let's calculate the number of nectar

TEMPERATURE: 0.7
system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?
assistant
Let's break down the problem step-by-step and then implement it in Python using sympy.

1. First, we calculate the tot

In [11]:
# USING A BIGGER MODEL
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval()

prompt = """Solve: There are 12 crates that each contain 150 oranges. 
There are 16 boxes that each hold 30 nectarines. 
How many pieces of fruit are in total?"""

messages = [{"role": "user", "content": prompt}]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

def generate_with_temperature(temp):
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=True,
            temperature=temp,
            top_p=0.95
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

for t in [0.3, 0.7, 1.0, 1.3]:
    print("\n" + "="*50)
    print("TEMPERATURE:", t)
    print("="*50)
    print(generate_with_temperature(t))

KeyboardInterrupt: 